<h1> Approximating value function with neural networks </h1>

In those two tutorials, we will be approximating state-action value function $Q(s,a)$ by a neural network. The environment we will use is the cart-pole environment of the OpenAI gym library. In this environment, the goal is to balance an inverse pendulum. Once the pendulum fails, the episode terminates. As long as the pendulum is more or less upright, you obtain reward $+1$ for each step. The environment is considered solved once you can balance the pendulum for $200$ steps or more. Actions are $0$ and $1$ for pushing the cart to the left or right. The state-space contains four continuous variables.

See [https://github.com/openai/gym/blob/master/gym/envs/classic_control/cartpole.py](https://github.com/openai/gym/blob/master/gym/envs/classic_control/cartpole.py) and [https://gymnasium.farama.org/environments/classic_control/cart_pole/](https://gymnasium.farama.org/environments/classic_control/cart_pole/) for more details.

In [2]:
!pip install matplotlib

^C


In [1]:
%matplotlib inline
!pip install gymnasium
import gymnasium as gym

import matplotlib
import matplotlib.pyplot as plt

env = gym.make('CartPole-v0')


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\gavul\Desktop\cvut_programming\CTU_SMU\lab04\venv_lab04\Lib\site-packages\gymnasium\envs\registration.py:512: DeprecationWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(


We will break the whole algorithm into several blocks we will implement separately. The first and the simplest one is the replay memory. Replay memory is a cyclic buffer that will be used to store triplets of state, action, and the sampled $Q(s,a)$. To speed up the work, the implementation is provided below.

In [2]:
class ReplayMemory:
  def __init__(self, capacity):
    # a cyclic buffer of a given size
    self.capacity = capacity
    self.memory = []
    self.head = 0 # will not be update if size < capacity, then we start from 0

  def put(self, state, action, q_state_action):
    # TD_target = q_state_action
    # store a sample into the buffer
    if(self.size() < self.capacity):
      self.memory.append((state, action, q_state_action))
    else:
      self.memory[self.head] = (state, action, q_state_action)
      self.head = (self.head + 1) % self.capacity

  def sample(self, number):
    # samples a given number of samples uniformly i.i.d. from the buffer
    return(random.sample(self.memory, number))

  def size(self):
    # gets the actual size of the buffer
    return len(self.memory)


In the next step, we need to create a simple neural network to approximate the $Q$-values. If you do not know how to create a simple neural network, try visiting, for example, [this tutorial](https://www.datacamp.com/tutorial/pytorch-tutorial-building-a-simple-neural-network-from-scratch).

In [3]:
from torch import nn

class Network(nn.Module):
  def __init__(self):
    super().__init__()
    self.hidden = nn.Linear(4, 256)
    self.hidden2 = nn.Linear(256, 256)
    self.output = nn.Linear(256, 2)

  def forward(self, x):
    # TODO implement the forward pass through the network.
    x = self.hidden(x)
    x = self.sigmoid(x)
    x = self.hidden2(x)
    x = self.sigmoid2(x)
    x = self.output(x)

    return x

In [9]:
from torch import nn

class Network(nn.Module):
  def __init__(self):
    super().__init__()
    self.hidden = nn.Linear()

  def forward(self, x):
    # TODO implement the forward pass through the network.
    x = self.hidden(x)

    # TODO

    return x

In [4]:
from torch import nn
import torch.nn.functional as F

input_dim = 4 # Four input parameters
output_dim = 2 # Two actions (left, right)

class Network(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(input_dim, 64)
    self.fc2 = nn.Linear(64, output_dim)

  def forward(self, x):
    # TODO implement the forward pass through the network.
    x = F.relu(self.fc1(x))
    return self.fc2(x)
    

Next, we will create a helper class <code>Model</code> that we will use to access the network. We will implement the greedy and $\varepsilon$-greedy policies, together with the optimization step.

In [ ]:
import torch
import numpy as np
import random

class Model:
  def __init__(self) -> None:
      super().__init__()
      self.network = Network()
      # we will need a proper loss function and an optimizer
      # you may get inspired here:
      # https://pytorch.org/tutorials/intermediate/reinforcement_q_learning.html
      # or select one from the list here:
      # https://neptune.ai/blog/pytorch-loss-functions
      # https://pytorch.org/docs/stable/optim.html
      self.criterion = # TODO choose your loss
      self.optimizer = # TODO chose your optimizer

  def greedy_policy(self, observation):
    # TODO implement the greedy policy, return 0/1
    with torch.no_grad():
      # TODO
      return 0

  def greedy_q_value(self, observation):
    # TODO implement the method to estimate U(s) as max_a Q(s,a)
    # we will need this method to calculate the target value to learn
    with torch.no_grad():
      return 0.0

  def eps_greedy(self, observation, epsilon):
    # TODO implement epsilon-greedy policy
    return 0

  def optimize_batch(self, replay_memory, batch_size):
    # in this method, we will actually train the neural network

    # if there are not enough samples in the history, skip learning
    if(replay_memory.size() < batch_size): return

    sample = replay_memory.sample(batch_size)

    prediction_vals = # TODO calculate predicted values by the netwrok on the sample
    targets = # TODO this is what the learning should have predicted

    # actual training
    loss = self.criterion(prediction_vals, targets)
    self.optimizer.zero_grad()
    loss.backward()

    for param in self.network.parameters():
      param.grad.data.clamp_(-1, 1)
    self.optimizer.step()

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

class Model:
    def __init__(self) -> None:
        self.network = Network()
        # MSE is standard for regression-like Q-value targets
        self.criterion = nn.MSELoss() 
        # Adam is generally more robust for RL than standard SGD
        self.optimizer = optim.Adam(self.network.parameters(), lr=0.001)

    def greedy_policy(self, observation):
        # Convert observation to tensor and add batch dimension
        obs_tensor = torch.tensor(observation, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            q_values = self.network(obs_tensor)
            # Pick the action with the highest Q-value
            return torch.argmax(q_values).item()

    def greedy_q_value(self, observation):
        # Used for the Bellman equation: max_a Q(s', a)
        obs_tensor = torch.tensor(observation, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            q_values = self.network(obs_tensor)
            return torch.max(q_values).item()

    def eps_greedy(self, observation, epsilon):
        if random.random() < epsilon:
            return env.action_space.sample() # Explore
        else:
            return self.greedy_policy(observation) # Exploit

    def optimize_batch(self, replay_memory, batch_size):
        if replay_memory.size() < batch_size: 
            return

        samples = replay_memory.sample(batch_size)
        # samples is a list of (state, action, td_target)
        
        states = torch.tensor([s[0] for s in samples], dtype=torch.float32)
        actions = torch.tensor([s[1] for s in samples], dtype=torch.long).unsqueeze(1)
        targets = torch.tensor([s[2] for s in samples], dtype=torch.float32).unsqueeze(1)

        # Get Q-values for the specific actions taken
        prediction_vals = self.network(states).gather(1, actions)

        loss = self.criterion(prediction_vals, targets)
        
        self.optimizer.zero_grad()
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        for param in self.network.parameters():
            if param.grad is not None:
                param.grad.data.clamp_(-1, 1)
        
        self.optimizer.step()


Finally, we are ready to put everything together in the OpenAI gym library.

In [ ]:
# TODO feel free to modify
NUMBER_OF_EPISODES = 1000
REPLAY_MEMORY_SIZE = 3000
BATCH_SIZE = 256

model = Model()
history = ReplayMemory(REPLAY_MEMORY_SIZE)

for e in range(NUMBER_OF_EPISODES):
  last_obs, _ = env.reset()

  for i in range(500): # the environment won't let us more than 500 actions
    epsilon = # TODO calculate the epsilon

    # do the step
    last_action = model.eps_greedy(last_obs, epsilon)
    obs, reward, done, _, _ = env.step(last_action)

    # store into history and optimize the model
    td_target = # TODO calculate the sampled value
    history.put(last_obs, last_action, td_target)

    model.optimize_batch(history, BATCH_SIZE)

    last_obs = obs

    # break and print how long we were able to balance the pendulum
    if done:
      print(i) # to know the length of the episode
      break

In [6]:
NUMBER_OF_EPISODES = 1000
REPLAY_MEMORY_SIZE = 10000
BATCH_SIZE = 64
GAMMA = 0.99  # Discount factor
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY = 0.995 # Decay per episode

model = Model()
history = ReplayMemory(REPLAY_MEMORY_SIZE)
epsilon = EPS_START

for e in range(NUMBER_OF_EPISODES):
    last_obs, _ = env.reset()
    total_reward = 0

    for i in range(500):
        # Step the environment
        last_action = model.eps_greedy(last_obs, epsilon)
        obs, reward, done, truncated, _ = env.step(last_action)
        
        # Calculate TD Target
        if done:
            td_target = reward # No future reward if episode ends
        else:
            # Bellman Equation: r + gamma * max_a Q(s', a)
            td_target = reward + GAMMA * model.greedy_q_value(obs)

        # Store and Train
        history.put(last_obs, last_action, td_target)
        model.optimize_batch(history, BATCH_SIZE)

        last_obs = obs
        total_reward += reward

        if done or truncated:
            break
            
    # Decay epsilon
    epsilon = max(EPS_END, epsilon * EPS_DECAY)
    
    if (e + 1) % 10 == 0:
        print(f"Episode {e+1}: Score = {total_reward}, Epsilon = {epsilon:.2f}")

C:\Users\gavul\AppData\Local\Temp\ipykernel_30312\244597885.py:43: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  states = torch.tensor([s[0] for s in samples], dtype=torch.float32)


Episode 10: Score = 23.0, Epsilon = 0.95
Episode 20: Score = 11.0, Epsilon = 0.90
Episode 30: Score = 23.0, Epsilon = 0.86
Episode 40: Score = 14.0, Epsilon = 0.82
Episode 50: Score = 27.0, Epsilon = 0.78
Episode 60: Score = 17.0, Epsilon = 0.74
Episode 70: Score = 24.0, Epsilon = 0.70
Episode 80: Score = 17.0, Epsilon = 0.67
Episode 90: Score = 29.0, Epsilon = 0.64
Episode 100: Score = 11.0, Epsilon = 0.61
Episode 110: Score = 12.0, Epsilon = 0.58
Episode 120: Score = 11.0, Epsilon = 0.55
Episode 130: Score = 19.0, Epsilon = 0.52
Episode 140: Score = 12.0, Epsilon = 0.50
Episode 150: Score = 10.0, Epsilon = 0.47
Episode 160: Score = 22.0, Epsilon = 0.45
Episode 170: Score = 11.0, Epsilon = 0.43
Episode 180: Score = 17.0, Epsilon = 0.41
Episode 190: Score = 11.0, Epsilon = 0.39
Episode 200: Score = 23.0, Epsilon = 0.37
Episode 210: Score = 12.0, Epsilon = 0.35
Episode 220: Score = 11.0, Epsilon = 0.33
Episode 230: Score = 11.0, Epsilon = 0.32
Episode 240: Score = 11.0, Epsilon = 0.30
E

## A second neural network?

Maybe you already found out that learning the neural network this way might not be very stable. We are trying to match the outputs to $Q$-values that are not static. This causes a drift in the expected outputs that might cause the neural network to encounter large errors between the sampled $Q$-value $r + \gamma \max_{a'} Q(s', a')$ and the predicted $Q(s,a)$ one. To stabilize the learning, a common approach is to include a second network with fixed $Q$-values to use as a target reference.

A very good explanation of why we need a second network is in the answer to this query:
https://stackoverflow.com/questions/54237327/why-is-a-target-network-required.
Also, an explanation of how the second network is used is here:
https://www.analyticsvidhya.com/blog/2019/04/introduction-deep-q-learning-python/, or in this example: https://pytorch.org/tutorials/intermediate/reinforcement_q_learning.html

Therefore, your next task is to modify your code to include a second network. Is learning faster? More stable?

In [7]:
import copy

class Model:
    def __init__(self, lr=0.001) -> None:
        self.network = Network()
        # Create a second network as the target
        self.target_network = Network()
        # Initialize it with the same weights
        self.target_network.load_state_dict(self.network.state_dict())
        self.target_network.eval() # Target network is always in eval mode

        self.criterion = nn.MSELoss()
        self.optimizer = optim.Adam(self.network.parameters(), lr=lr)

    def synchronize(self):
        """Update the target network with the policy network weights."""
        self.target_network.load_state_dict(self.network.state_dict())

    def greedy_policy(self, observation):
        obs_tensor = torch.tensor(observation, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            # Decisions are always made by the main network
            q_values = self.network(obs_tensor)
            return torch.argmax(q_values).item()

    def greedy_q_value(self, observation):
        """Estimate max_a Q(s', a) using the TARGET network."""
        obs_tensor = torch.tensor(observation, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            # We use the target_network here to provide stable targets
            q_values = self.target_network(obs_tensor)
            return torch.max(q_values).item()

    def eps_greedy(self, observation, epsilon):
        if random.random() < epsilon:
            return env.action_space.sample()
        return self.greedy_policy(observation)

    def optimize_batch(self, replay_memory, batch_size):
        if replay_memory.size() < batch_size: 
            return

        samples = replay_memory.sample(batch_size)
        states = torch.tensor([s[0] for s in samples], dtype=torch.float32)
        actions = torch.tensor([s[1] for s in samples], dtype=torch.long).unsqueeze(1)
        targets = torch.tensor([s[2] for s in samples], dtype=torch.float32).unsqueeze(1)

        # Policy network predicts current Q-values
        prediction_vals = self.network(states).gather(1, actions)

        loss = self.criterion(prediction_vals, targets)
        self.optimizer.zero_grad()
        loss.backward()
        
        for param in self.network.parameters():
            if param.grad is not None:
                param.grad.data.clamp_(-1, 1)
        self.optimizer.step()

## Challenge 1 - learn from images

Deep reinforcement learning is often connected with learning from images directly. Our state space might be formed by the difference of images of the cart pole between the two episodes. Basically, we might capture the image of the screen in two consecutive steps and pass their difference to a **convolutional neural network** to predict the next movement. The difference captures information about the velocity of the cart, the angular velocity of the pendulum, and positional information. Your task is to take your code and modify it to work this way. Of course, you do not need to implement everything yourself; for example, feel free to copy method <code>get_screen</code> (and take other inspiration) from https://pytorch.org/tutorials/intermediate/reinforcement_q_learning.html.

In [9]:
NUMBER_OF_EPISODES = 1000
REPLAY_MEMORY_SIZE = 10000
BATCH_SIZE = 128
GAMMA = 0.99
TARGET_UPDATE_FREQ = 10 # Update target network every 10 episodes
EPS_START = 1.0
EPS_END = 0.01
EPS_DECAY = 0.99

model = Model(lr=0.0005)
history = ReplayMemory(REPLAY_MEMORY_SIZE)
epsilon = EPS_START

for e in range(NUMBER_OF_EPISODES):
    last_obs, _ = env.reset()
    total_reward = 0

    for i in range(500):
        last_action = model.eps_greedy(last_obs, epsilon)
        obs, reward, done, truncated, _ = env.step(last_action)

        # Use the Model's greedy_q_value (which now uses the target network)
        if done:
            td_target = reward
        else:
            td_target = reward + GAMMA * model.greedy_q_value(obs)

        history.put(last_obs, last_action, td_target)
        model.optimize_batch(history, BATCH_SIZE)

        last_obs = obs
        total_reward += reward

        if done or truncated:
            break
            
    # Periodically sync the target network
    if (e + 1) % TARGET_UPDATE_FREQ == 0:
        model.synchronize()

    epsilon = max(EPS_END, epsilon * EPS_DECAY)
    
    if (e + 1) % 20 == 0:
        print(f"Episode {e+1}: Score = {total_reward}, Epsilon = {epsilon:.2f}")

Episode 20: Score = 17.0, Epsilon = 0.82
Episode 40: Score = 13.0, Epsilon = 0.67
Episode 60: Score = 12.0, Epsilon = 0.55
Episode 80: Score = 23.0, Epsilon = 0.45
Episode 100: Score = 9.0, Epsilon = 0.37
Episode 120: Score = 12.0, Epsilon = 0.30
Episode 140: Score = 10.0, Epsilon = 0.24
Episode 160: Score = 12.0, Epsilon = 0.20
Episode 180: Score = 10.0, Epsilon = 0.16
Episode 200: Score = 10.0, Epsilon = 0.13
Episode 220: Score = 9.0, Epsilon = 0.11
Episode 240: Score = 9.0, Epsilon = 0.09
Episode 260: Score = 12.0, Epsilon = 0.07
Episode 280: Score = 9.0, Epsilon = 0.06
Episode 300: Score = 9.0, Epsilon = 0.05
Episode 320: Score = 10.0, Epsilon = 0.04
Episode 340: Score = 10.0, Epsilon = 0.03
Episode 360: Score = 9.0, Epsilon = 0.03
Episode 380: Score = 8.0, Epsilon = 0.02
Episode 400: Score = 10.0, Epsilon = 0.02
Episode 420: Score = 8.0, Epsilon = 0.01
Episode 440: Score = 12.0, Epsilon = 0.01
Episode 460: Score = 9.0, Epsilon = 0.01
Episode 480: Score = 9.0, Epsilon = 0.01
Episod

In [10]:
class ReplayMemory:
    def __init__(self, capacity):
        self.capacity = capacity
        self.memory = []
        self.head = 0

    def put(self, state, action, reward, next_state, done):
        # Store the raw transition ingredients
        if len(self.memory) < self.capacity:
            self.memory.append((state, action, reward, next_state, done))
        else:
            self.memory[self.head] = (state, action, reward, next_state, done)
            self.head = (self.head + 1) % self.capacity

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def size(self):
        return len(self.memory)

In [11]:
class Model:
    def __init__(self, lr=0.001):
        self.network = Network()
        self.target_network = Network()
        self.target_network.load_state_dict(self.network.state_dict())
        
        self.optimizer = torch.optim.Adam(self.network.parameters(), lr=lr)
        self.criterion = nn.MSELoss()

    def synchronize(self):
        self.target_network.load_state_dict(self.network.state_dict())

    def optimize_batch(self, replay_memory, batch_size, gamma):
        if replay_memory.size() < batch_size:
            return

        batch = replay_memory.sample(batch_size)
        
        # Unpack batch and convert to tensors
        states = torch.tensor([b[0] for b in batch], dtype=torch.float32)
        actions = torch.tensor([b[1] for b in batch], dtype=torch.long).unsqueeze(1)
        rewards = torch.tensor([b[2] for b in batch], dtype=torch.float32).unsqueeze(1)
        next_states = torch.tensor([b[3] for b in batch], dtype=torch.float32)
        dones = torch.tensor([b[4] for b in batch], dtype=torch.float32).unsqueeze(1)

        # 1. Get current Q values from policy network
        current_q = self.network(states).gather(1, actions)

        # 2. Get max Q values for next states from TARGET network
        with torch.no_grad():
            max_next_q = self.target_network(next_states).max(1)[0].unsqueeze(1)
            # Bellman Equation: If done, target is just reward. Otherwise reward + gamma * max_next_q
            targets = rewards + (gamma * max_next_q * (1 - dones))

        # 3. Compute loss and update
        loss = self.criterion(current_q, targets)
        self.optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping is very important for stability in RL
        torch.nn.utils.clip_grad_norm_(self.network.parameters(), 1.0)
        self.optimizer.step()

    def eps_greedy(self, state, epsilon):
        if random.random() < epsilon:
            return random.randint(0, 1) # Explore
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            return self.network(state_t).argmax().item() # Exploit

In [12]:
# Hyperparameters
GAMMA = 0.99
BATCH_SIZE = 64
LR = 0.001
TARGET_UPDATE_FREQ = 10 # episodes
EPS_DECAY = 0.995 # Slower decay
REPLAY_SIZE = 10000

model = Model(lr=LR)
memory = ReplayMemory(REPLAY_SIZE)
epsilon = 1.0

for e in range(1000):
    state, _ = env.reset()
    episode_reward = 0
    
    for t in range(500):
        action = model.eps_greedy(state, epsilon)
        next_state, reward, done, truncated, _ = env.step(action)
        
        # Store raw transition
        memory.put(state, action, reward, next_state, done)
        
        # Train
        model.optimize_batch(memory, BATCH_SIZE, GAMMA)
        
        state = next_state
        episode_reward += reward
        
        if done or truncated:
            break
            
    # Update Target Network
    if (e + 1) % TARGET_UPDATE_FREQ == 0:
        model.synchronize()
        
    epsilon = max(0.01, epsilon * EPS_DECAY)
    
    if (e + 1) % 10 == 0:
        print(f"Episode {e+1}: Score: {episode_reward}, Epsilon: {epsilon:.2f}")

Episode 10: Score: 13.0, Epsilon: 0.95
Episode 20: Score: 21.0, Epsilon: 0.90
Episode 30: Score: 16.0, Epsilon: 0.86
Episode 40: Score: 15.0, Epsilon: 0.82
Episode 50: Score: 50.0, Epsilon: 0.78
Episode 60: Score: 122.0, Epsilon: 0.74
Episode 70: Score: 75.0, Epsilon: 0.70
Episode 80: Score: 21.0, Epsilon: 0.67
Episode 90: Score: 20.0, Epsilon: 0.64
Episode 100: Score: 65.0, Epsilon: 0.61
Episode 110: Score: 200.0, Epsilon: 0.58
Episode 120: Score: 55.0, Epsilon: 0.55
Episode 130: Score: 138.0, Epsilon: 0.52
Episode 140: Score: 51.0, Epsilon: 0.50
Episode 150: Score: 200.0, Epsilon: 0.47
Episode 160: Score: 200.0, Epsilon: 0.45
Episode 170: Score: 200.0, Epsilon: 0.43
Episode 180: Score: 73.0, Epsilon: 0.41
Episode 190: Score: 200.0, Epsilon: 0.39
Episode 200: Score: 200.0, Epsilon: 0.37
Episode 210: Score: 200.0, Epsilon: 0.35
Episode 220: Score: 200.0, Epsilon: 0.33
Episode 230: Score: 200.0, Epsilon: 0.32
Episode 240: Score: 107.0, Epsilon: 0.30
Episode 250: Score: 164.0, Epsilon: 0

## Challenge 2 - double inverted pendulum

If you do not like the previous challenge, you might try a different one - a double inverted pendulum. The problem is very similar to the inverted pendulum; however, it is much harder to balance the double pendulum. Also, the state space is much larger. Therefore, you will need a larger network and more computational time.

See https://github.com/openai/gym/blob/master/gym/envs/mujoco/inverted_double_pendulum.py for documentation. Note that the action is continuous; you will need to modify your network to account for infinite action space.

In [ ]:
!apt-get install -y \
    libgl1-mesa-dev \
    libgl1-mesa-glx \
    libglew-dev \
    libosmesa6-dev \
    software-properties-common

!apt-get install -y patchelf

!pip install free-mujoco-py
import mujoco_py

env = gym.make('InvertedDoublePendulum-v2')